# Import

In [2]:
%pip install tensorflow keras transformer


ERROR: Could not find a version that satisfies the requirement transformer (from versions: none)
ERROR: No matching distribution found for transformer


In [3]:
%pip install keras-tuner


Note: you may need to restart the kernel to use updated packages.


In [43]:
%load_ext autoreload
%autoreload 2

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
%pip install hmmlearn
%pip install pgmpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install keras-nlp --upgrade

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install tf-keras

Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install matplotlib


In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm import tqdm
import keras_tuner as kt
from tensorflow.keras.models import load_model
import keras_nlp


e:\anaconda3\envs\ml_env_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score

In [11]:
!pip install xgboost


In [12]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import hmmlearn.hmm
from hmmlearn.hmm import GaussianHMM
from sklearn_crfsuite import CRF
from sklearn.metrics import log_loss, hinge_loss, precision_score, recall_score, f1_score, roc_auc_score

In [13]:
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

In [14]:
## Options
pd.set_option("max_colwidth", None)

In [15]:
# Get the absolute path to the 'src' directory
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(project_root)
print(project_root)

e:\2_LEARNING_BKU\2_File_2\K22_HK242\CO3117_Machine_Learning\Main


In [44]:
from src.features.build_features_utils import *  # Assuming build_features_utils is inside build_features.py
from src.models.models_utils import *  # Assuming utils.py exists inside src/models/

# Dict

In [17]:
# Dictionary for models
MODEL_DICT = {
    "decision_tree": DecisionTreeClassifier,
    "perceptron": Perceptron,
    "mlp": MLPClassifier,
    "bayesian": GaussianNB,
    "random_forest": RandomForestClassifier,
    "xgboost": xgb.XGBClassifier,
    "logistic_regression": LogisticRegression,
    "svm": SVC,
    "lda": LDA
} 

# Dictionary for model parameters
MODEL_PARAMS = {
    "lda": {
        "solver": ["lsqr", "eigen"],
        "shrinkage": [None, "auto"],  # Only used with 'lsqr' or 'eigen'
        "tol": [1e-4, 1e-3, 1e-2]     # Tolerance for convergence
    },
    
    # "decision_tree": {
    #     "criterion": ["gini", "entropy"],
    #     "max_depth": [10, 20],
    #     "min_samples_split": [2, 5],
    #     "min_samples_leaf": [1, 2],
    #     "max_features": ["sqrt", "log2"]
    # },
    
    "decision_tree": {
        "criterion": ["gini", "entropy"],
        "max_depth": [10, 20, 30, 40],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },
    
    # "perceptron": {
    #     "max_iter": [1000, 2000],
    #     "tol": [1e-3],
    #     "eta0": [0.001],
    #     "penalty": ["l2"],
    #     "alpha": [0.0001, 0.001]
    # },
    
    "perceptron": {
        "max_iter": [1000, 2000],
        "tol": [1e-3, 1e-4],
        "eta0": [0.001, 0.01, 0.1],
        "penalty": [None, "l2", "l1"],
        "alpha": [0.0001, 0.001, 0.01]
    },
    
    "mlp": {
        "hidden_layer_sizes": [(100,)],
        "activation": ["tanh", "logistic"],
        "solver": ["sgd"],
        "alpha": [0.01],
        "batch_size": [32],
        "max_iter": [2000],
    },
    
    # "mlp": {
    #     "hidden_layer_sizes": [(50,), (100,), (50, 50), (100, 100)],
    #     "activation": ["relu", "tanh", "logistic"],
    #     "solver": ["adam", "sgd"],
    #     "alpha": [0.0001, 0.001, 0.01],
    #     "batch_size": [32, 64, 128],
    #     "max_iter": [500, 1000],
    #     "learning_rate": ["constant", "invscaling", "adaptive"]
    # },
    
    "bayesian": {
        "priors": [None, [0.5, 0.5], [0.4, 0.6], [0.3, 0.7], [0.2, 0.8], [0.1, 0.9], [0.05, 0.95]],
        "var_smoothing": [1e-9, 1e-8, 1e-7]
    },
    
    "random_forest": {
        "n_estimators": [100, 200],
        "max_depth": [10],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"],
        "bootstrap": [True, False]
    },
    
    # "random_forest": {
    #     "n_estimators": [50, 100, 200],
    #     "max_depth": [None, 10, 20, 30],
    #     "min_samples_split": [2, 5, 10],
    #     "min_samples_leaf": [1, 2, 4],
    #     "max_features": ["auto", "sqrt", "log2"],
    #     "bootstrap": [True, False]
    # },
    
    "xgboost": {
        "n_estimators": [100],
        "learning_rate": [0.01, 0.1],
        "max_depth": [6, 10]
    },
    
    # "xgboost": {
    #     "n_estimators": [100, 200, 300],
    #     "learning_rate": [0.01, 0.1, 0.2],
    #     "max_depth": [3, 6, 10],
    #     "subsample": [0.8, 1.0],
    #     "colsample_bytree": [0.8, 1.0],
    #     "gamma": [0, 0.1, 0.2]
    # },
    
    "svm": {
        "kernel": ["linear"],
        "C": [0.001, 0.01, 0.1, 1],
        "gamma": [0.1, 0.01, "scale", "auto"]
    },
    
    # "svm": {
    #     "kernel": ["linear", "rbf", "poly"],
    #     "C": [0.1, 1, 10, 100],
    #     "gamma": [0.1, 0.01, "scale", "auto"],
    #     "degree": [2, 3, 4]
    # },
    
    # "logistic_regression": {
    #     "penalty": ["l2"],
    #     "C": [0.1, 1.0],
    #     "max_iter": [1000, 2000]
    # },
    
    "logistic_regression": {
        "penalty": ["l1", "l2", "elasticnet", None],
        "C": [0.1, 1.0, 10.0],
        "max_iter": [1000, 2000]
    },
    
    
    # "hmm": {
    #     "n_components": [2],  # Keep it small
    #     "covariance_type": ["diag"],  # Simpler covariance type
    #     "n_iter": [500],  # Reduce iterations
    #     "init_params": ["stmc"],  # Initialize start probabilities, transition matrix, and means/covariance
    #     "params": ["stmc"]
    # },
    
    "hmm": {
        "n_components": [2, 3, 4],
        "covariance_type": ["diag", "full", "tied"],
        "n_iter": [100, 200],
        "init_params": ["c", "s", "cs"],
        "params": ["c", "t", "ct"]
    },
    
    "bayes_network": {
        "structure": [None],
        "n_bins": [2],
        "strategy": ["kmeans"],
        "min_unique_values": [2],
        "max_features": [10]
    },
    
    # "crf": {
    #     "c1": [0.1, 0.01],  # L1 Regularization
    #     "c2": [0.1, 0.01],  # L2 Regularization
    #     "max_iterations": [50, 100]  # Limit iterations
    # }
}

BEST_MODEL_PARAMS = {
    "decision_tree": {
        "criterion": "gini",
        "max_depth": 40,
        "min_samples_split": 10,
        "min_samples_leaf": 4,
        "max_features": "sqrt"
    },
    
    "perceptron": {
        "max_iter": 1000,
        "tol": 1e-3,
        "eta0": 0.001,
        "penalty": "l2",
        "alpha": 0.0001
    },
    
    "mlp": {
        "hidden_layer_sizes": (100,),
        "activation": "logistic",
        "solver": "sgd",
        "alpha": 0.01,
        "batch_size": 32,
        "max_iter": 2000,
    },
    
    "bayesian": {
        "priors": [0.3, 0.7],
        "var_smoothing": 1e-9
    },
    
    "random_forest": {
        "n_estimators": [100],
        "max_depth": [10],
        "min_samples_split": [5],
        "min_samples_leaf": [1],
        "max_features": ["sqrt"]
    },
    
    "xgboost": {
        "n_estimators": 150,
        "learning_rate": 0.1,
        "max_depth": 15
    },
    
    "svm": {
        "kernel": ["linear"],
        "C": [0.001, 0.01, 0.1, 1],
        "gamma": [0.1, 0.01, "scale", "auto"]
    },
    
    "logistic_regression": {
        "penalty": "l2",
        "C": 0.1,
        "max_iter": 1000
    },
    
    "hmm": {
        "n_components": [2, 3, 4],
        "covariance_type": ["diag", "full", "tied"],
        "n_iter": [100, 200],
        "init_params": ["c", "s", "cs"],
        "params": ["c", "t", "ct"]
    },
    
    "bayes_network": {
        "structure": [None],
        "n_bins": [2],
        "strategy": ["kmeans"],
        "min_unique_values": [2],
        "max_features": [10]
    },
    
    # "crf": {
    #     "c1": [0.1, 0.01],  # L1 Regularization
    #     "c2": [0.1, 0.01],  # L2 Regularization
    #     "max_iterations": [50, 100]  # Limit iterations
    # }
}

# Dictionary for dimensionality reduction methods
DIMENSIONALITY_REDUCTION_DICT = {
    "pca": PCA,
    "lda": LDA,
}

# Load dataset

In [18]:
# Load dataset
dataset_path = os.path.join(project_root, "data", "final", "final_clean_no_neutral_no_duplicates_v1.csv")
df = pd.read_csv(dataset_path)


In [19]:
df.head()

,target,text,text_clean,text_length,text_clean_length
0,0.0,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D",switchfoot awww thats bummer shoulda got david carr third day,19,10
1,0.0,is upset that he can't update his Facebook by texting it... and might cry as a result School today also. Blah!,upset cant update facebook texting might cry result school today also blah,21,12
2,0.0,@Kenichan I dived many times for the ball. Managed to save 50% The rest go out of bounds,kenichan dived many times ball managed save rest go bounds,18,10
3,0.0,my whole body feels itchy and like its on fire,whole body feels itchy like fire,10,6
4,0.0,"@nationwideclass no, it's not behaving at all. i'm mad. why am i here? because I can't see you all over there.",nationwideclass behaving im mad cant see,21,6


In [20]:
# Replace target 4 with 1
df["target"] = df["target"].replace(4, 1)


# Build features

## Defined

In [21]:
# feature_methods = ["tfidf", "count", "word2vec", "glove"]
feature_methods = ["count"]
df_sampled = df.sample(n=1000, random_state=42)

In [22]:
doc_lst = df_sampled["text_clean"].tolist()
label_lst = df_sampled["target"].tolist()

In [23]:
X_train_features_dict, X_test_features_dict, y_train, y_test = build_vector_for_text(df_sampled, feature_methods, project_root)


🔎 Running feature extraction...



Feature Extraction Progress: 100%|██████████| 1/1 [00:00<00:00,  7.41it/s]


🔍 Processing feature extraction using: count...
✅ count - Train shape: (800, 2000), Test shape: (200, 2000)


In [24]:
print("\n📊 Dataset Shapes:")

# Print the shape of feature matrices for each feature method
for feature_method, X_train in X_train_features_dict.items():
    print(f"🔹 X_train ({feature_method}): {X_train.shape}")
    
for feature_method, X_test in X_test_features_dict.items():
    print(f"🔹 X_test ({feature_method}): {X_test.shape}")

# Print y_train and y_test shapes
print(f"\n🎯 y_train shape: {y_train.shape}")
print(f"🎯 y_test shape: {y_test.shape}")



📊 Dataset Shapes:
🔹 X_train (count): (800, 2000)
🔹 X_test (count): (200, 2000)

🎯 y_train shape: (800,)
🎯 y_test shape: (200,)


## Test PCA (reduce dim) - LDA (classifier) => done

In [25]:
X_train_pca_dict, X_test_pca_dict, y_train, y_test = build_vector_for_text(df_sampled, feature_methods, project_root, "pca", 100)


🔎 Running feature extraction...



Feature Extraction Progress:   0%|          | 0/1 [00:00<?, ?it/s]


🔍 Processing feature extraction using: count...


Feature Extraction Progress: 100%|██████████| 1/1 [00:04<00:00,  4.38s/it]

✅ count - Train shape: (800, 100), Test shape: (200, 100)


In [26]:
print("\n📊 Dataset Shapes:")

# Print the shape of feature matrices for each feature method
for feature_method, X_train in X_train_pca_dict.items():
    print(f"🔹 X_train ({feature_method}): {X_train.shape}")
    
for feature_method, X_test in X_test_pca_dict.items():
    print(f"🔹 X_test ({feature_method}): {X_test.shape}")

# Print y_train and y_test shapes
print(f"\n🎯 y_train shape: {y_train.shape}")
print(f"🎯 y_test shape: {y_test.shape}")



📊 Dataset Shapes:
🔹 X_train (count): (800, 100)
🔹 X_test (count): (200, 100)

🎯 y_train shape: (800,)
🎯 y_test shape: (200,)


## Test cho cac api call moi trong feature builder => done

In [27]:
# # Test each feature selection method
# feature_selections = ["variance", "chi2", "topic_modeling", None]
# for feature_selection in feature_selections:
#     print(f"\n🔬 Testing with feature_selection: {feature_selection or 'None'}")
    
#     # Build features with current feature selection method
#     X_train_features_dict, X_test_features_dict, y_train, y_test = build_vector_for_text(
#         df_sampled=df_sampled,
#         feature_methods=feature_methods,
#         project_root=project_root,
#         reduce_dim="pca",  # Adding PCA as an example, can be None or "lda"
#         n_components=50,
#         feature_selection=feature_selection
#     )

#     print("\n📊 Dataset Shapes:")
    
#     # Print the shape of feature matrices for each feature method
#     for feature_method, X_train in X_train_features_dict.items():
#         print(f"🔹 X_train ({feature_method}): {X_train.shape}")
        
#     for feature_method, X_test in X_test_features_dict.items():
#         print(f"🔹 X_test ({feature_method}): {X_test.shape}")

#     # Print y_train and y_test shapes
#     print(f"\n🎯 y_train shape: {y_train.shape}")
#     print(f"🎯 y_test shape: {y_test.shape}")
    
#     print("-" * 50)

Variance 

=> count & glove 

=> not tfidf because No feature in X meets the variance threshold 0.01000. Skipping this method.

=> not w2v as n_components=50 must be between 0 and min(n_samples, n_features)=1 with svd_solver='covariance_eigh'. Skipping this method.

chi2 

=> can be with tfidf and count 

=> not w2v + glove as Input X must be non-negative.. Skipping this method.

topic_modeling

=> can be with tfidf and count 

=> not w2v + glove as Negative values in data passed to LatentDirichletAllocation.fit

Negative values in data passed to LatentDirichletAllocation.fit

## Test the Voting Classifier => done

In [28]:
# SELECTED_MODEL_DICT = {
#     "logistic_regression": LogisticRegression,
#     "xgboost": xgb.XGBClassifier,
#     "mlp": MLPClassifier,
#     "bayesian": GaussianNB,
# }

In [29]:
# # Call the function with all models (one of each)
# feature_use = "count"

# voting_clf = train_voting_classifier(
#     model_dict=SELECTED_MODEL_DICT, 
#     param_dict=BEST_MODEL_PARAMS, 
#     feature_method=feature_use, 
#     X=X_train_features_dict[feature_use], 
#     y=y_train, 
#     voting_type='soft', 
#     model_save_path="voting_model_all_models.pkl"
# )


In [30]:
# # Load the trained VotingClassifier model from the file
# voting_clf = joblib.load("voting_model_all_models.pkl")
# print("✅ Model loaded successfully from 'voting_model_all_models.pkl'")

In [31]:
# # Test the trained model on the test set
# y_pred = voting_clf.predict(X_test_features_dict[feature_use])

# # Print evaluation metrics
# print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
# print(f"ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")
# print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
# print(f"Precision: {precision_score(y_test, y_pred):.4f}")
# print(f"Recall: {recall_score(y_test, y_pred):.4f}")

# # Print Classification Report
# print("\n📄 Classification Report:\n")
# print(classification_report(y_test, y_pred))

In [32]:
# # Correct way to create multiple Logistic Regression models (passing classes, not objects)
# multiple_lr_models = {
#     f"logistic_regression_{i}": LogisticRegression  # Only pass the class, not the instance
#     for i in range(5)
# }

# # Provide a parameter dictionary
# lr_params = {
#     f"logistic_regression_{i}": {"penalty": "l2", "C": 0.1, "max_iter": 1000, "random_state": i}
#     for i in range(5)
# }

# # Call the function with multiple logistic regression models
# voting_clf = train_voting_classifier(
#     model_dict=multiple_lr_models, 
#     param_dict=lr_params,  # Now we provide the parameter dictionary
#     feature_method=feature_use, 
#     X=X_train_features_dict[feature_use], 
#     y=y_train, 
#     voting_type='soft', 
#     model_save_path="voting_model_multiple_lr.pkl"
# )

# # Test the trained model on the test set
# y_pred = voting_clf.predict(X_test_features_dict[feature_use])

# # Print evaluation metrics
# print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
# print(f"ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")
# print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
# print(f"Precision: {precision_score(y_test, y_pred):.4f}")
# print(f"Recall: {recall_score(y_test, y_pred):.4f}")

# # Print Classification Report
# print("\n📄 Classification Report:\n")
# print(classification_report(y_test, y_pred))

## Test Stacking => done

In [33]:
# # Define multiple Logistic Regression models (as classes, not instances)
# multiple_lr_models = {
#     f"logistic_regression_{i}": LogisticRegression  # Note: Passing the class, not instance
#     for i in range(5)
# }

# # Provide parameters for each logistic regression model
# lr_params = {
#     f"logistic_regression_{i}": {"penalty": "l2", "C": 0.1, "max_iter": 1000, "random_state": i}
#     for i in range(5)
# }

# # Train Stacking Classifier
# stacking_clf = train_stacking_classifier(
#     model_dict=multiple_lr_models,
#     param_dict=lr_params,
#     feature_method=feature_use,
#     X=X_train_features_dict[feature_use],
#     y=y_train,
#     final_estimator=LogisticRegression(),
#     model_save_path="stacking_model_multiple_lr.pkl"
# )

# # Test the trained model on the test set
# y_pred = stacking_clf.predict(X_test_features_dict[feature_use])

# # Print evaluation metrics
# print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
# print(f"ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")
# print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
# print(f"Precision: {precision_score(y_test, y_pred):.4f}")
# print(f"Recall: {recall_score(y_test, y_pred):.4f}")

# # Print Classification Report
# print("\n📄 Classification Report:\n")
# print(classification_report(y_test, y_pred))


In [34]:
# stacking_clf = train_stacking_classifier(
#     model_dict=SELECTED_MODEL_DICT,
#     param_dict=BEST_MODEL_PARAMS,
#     feature_method=feature_use,
#     X=X_train_features_dict[feature_use],
#     y=y_train,
#     final_estimator=LogisticRegression(),
#     model_save_path="stacking_model_selected_models.pkl"
# )

# # Test the trained model on the test set
# y_pred = stacking_clf.predict(X_test_features_dict[feature_use])

# # Print evaluation metrics
# print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
# print(f"ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")
# print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
# print(f"Precision: {precision_score(y_test, y_pred):.4f}")
# print(f"Recall: {recall_score(y_test, y_pred):.4f}")

# # Print Classification Report
# print("\n📄 Classification Report:\n")
# print(classification_report(y_test, y_pred))

## New test 

# New API Call

In [35]:
model_name_lst = [
    # "decision_tree",
    # "random_forest",
    # "xgboost", 
    # "perceptron", 
    # "mlp", 
    # "lstm",
    # "bayesian",
    # "GA",
    # "hmm",
    # "bayesnet",
    # "logistic_regression",
    # "svm",
    # "lda",
    # "bilstm", # not test ever
    # "bert" # not test ever ,
    "CRF"
]

In [36]:
trained_model = os.path.join(project_root, "src", "models")

In [37]:
# %pip uninstall tf-nightly
# %pip install tensorflow


In [38]:
# %pip show keras-nlp


In [ ]:
train_general_model(df_sampled, doc_lst, label_lst, model_name_lst, feature_methods, MODEL_DICT, MODEL_PARAMS, X_train_features_dict, X_test_features_dict, y_train, y_test)

[I 2025-04-12 13:30:08,178] A new study created in memory with name: no-name-4eb27123-8544-4f6c-9aea-c46195e43e0d



🔎 Running feature extraction and model training loop...


🚀 Training CRF models...

🖥️ Using device: cpu
🔍 Tuning hyperparameters...


In [40]:
predict_general_model(model_name_lst, feature_methods, X_test_features_dict, y_test, trained_model)

⚙️  Using device: cpu
Already trained and tested model: CRF


In [ ]:
train_general_model(df_sampled, doc_lst, label_lst, model_name_lst, feature_methods, MODEL_DICT, MODEL_PARAMS, X_train_pca_dict, X_test_pca_dict, y_train, y_test)


🔎 Running feature extraction and model training loop...


🚀 Training CRF models...

🖥️ Using device: cpu


[I 2025-04-12 13:00:02,378] A new study created in memory with name: no-name-72746071-016e-4050-8c4e-ed9b82814647


🔍 Tuning hyperparameters...


[I 2025-04-12 13:00:31,308] Trial 0 finished with value: 0.6721991701244814 and parameters: {'hidden_dim': 64, 'lr': 7.57742099727808e-05}. Best is trial 0 with value: 0.6721991701244814.
[I 2025-04-12 13:00:39,776] Trial 1 finished with value: 0.6721991701244814 and parameters: {'hidden_dim': 256, 'lr': 0.00041329957943100914}. Best is trial 0 with value: 0.6721991701244814.
[I 2025-04-12 13:00:41,836] Trial 2 finished with value: 0.6721991701244814 and parameters: {'hidden_dim': 64, 'lr': 3.222151287041304e-05}. Best is trial 0 with value: 0.6721991701244814.
[I 2025-04-12 13:00:48,012] Trial 3 finished with value: 0.0 and parameters: {'hidden_dim': 256, 'lr': 0.0004697034242299737}. Best is trial 0 with value: 0.6721991701244814.
[I 2025-04-12 13:00:51,068] Trial 4 finished with value: 0.6721991701244814 and parameters: {'hidden_dim': 64, 'lr': 0.0001150391148067497}. Best is trial 0 with value: 0.6721991701244814.


Epoch 1/30 - Train: 0.6924, Val: 0.6935
Epoch 2/30 - Train: 0.6909, Val: 0.6944
Epoch 3/30 - Train: 0.6905, Val: 0.6942
Epoch 4/30 - Train: 0.6906, Val: 0.6950
Epoch 5/30 - Train: 0.6905, Val: 0.6946
Epoch 6/30 - Train: 0.6904, Val: 0.6946
Epoch 7/30 - Train: 0.6903, Val: 0.6944
Epoch 8/30 - Train: 0.6904, Val: 0.6949
Epoch 9/30 - Train: 0.6902, Val: 0.6948
Epoch 10/30 - Train: 0.6902, Val: 0.6948
Epoch 11/30 - Train: 0.6906, Val: 0.6943
Epoch 12/30 - Train: 0.6903, Val: 0.6950
Epoch 13/30 - Train: 0.6903, Val: 0.6942
Epoch 14/30 - Train: 0.6900, Val: 0.6944
Epoch 15/30 - Train: 0.6904, Val: 0.6955
Epoch 16/30 - Train: 0.6900, Val: 0.6950
Epoch 17/30 - Train: 0.6904, Val: 0.6941
Epoch 18/30 - Train: 0.6901, Val: 0.6950
Epoch 19/30 - Train: 0.6899, Val: 0.6947
Epoch 20/30 - Train: 0.6899, Val: 0.6944
Epoch 21/30 - Train: 0.6900, Val: 0.6949
Epoch 22/30 - Train: 0.6898, Val: 0.6944
Epoch 23/30 - Train: 0.6898, Val: 0.6946
Epoch 24/30 - Train: 0.6901, Val: 0.6943
Epoch 25/30 - Train: 0.69

e:\2_LEARNING_BKU\2_File_2\K22_HK242\CO3117_Machine_Learning\Main\src\models\models_utils.py:1461: UserWarning: Glyph 128201 (\N{CHART WITH DOWNWARDS TREND}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
e:\2_LEARNING_BKU\2_File_2\K22_HK242\CO3117_Machine_Learning\Main\src\models\models_utils.py:1462: UserWarning: Glyph 128201 (\N{CHART WITH DOWNWARDS TREND}) missing from font(s) DejaVu Sans.
  plt.savefig("crf_loss_curve.png")


In [42]:
predict_general_model(model_name_lst, feature_methods, X_test_pca_dict, y_test, trained_model)

⚙️  Using device: cpu
Already trained and tested model: CRF
